# ColdSite-DTI — the 36-run audit grid (DAVIS, binary, Kaggle T4 x2)

**3 models x 4 splits x 3 seeds = 36 runs**, DAVIS, binary task. This is the scientific
spine of the paper as scoped in STATUS.md on 2026-08-18: every number in the ladder, the
faithfulness figure, the Holm-corrected audit table and the non-kinase control comes from
these checkpoints.

| model | role | cells |
|---|---|---|
| DeepDTA | accuracy anchor (no attention, never audited) | 12 |
| ColdSite-DTI | our model | 12 |
| HyperAttentionDTI | the published model under audit | 12 |

**Why binary.** HyperAttentionDTI and MolTrans only predict binding / non-binding, so the
audit table can only compare models on one metric -- AUROC -- if every model does the same
task. A pair counts as binding at **DAVIS pKd >= 7.0**, DeepDTA's published threshold,
defined once in `src/model/dataset.py` and shared by all three trainers. About 8% of DAVIS
pairs bind, so accuracy is ~0.92 for a model that always says "no": read AUROC, not accuracy.

The 12 DAVIS *regression* cells trained earlier do not count here -- different task.

## Before you run

| Setting | Value |
|---|---|
| Accelerator | **GPU T4 x2** |
| Internet | **On** |
| Environment | **Pin to original** |
| How to run | **Save Version -> Save & Run All (Commit)** |

## The loop you will repeat -- read this

Every commit starts in a **fresh, empty container**. Finished cells do not carry over by
themselves (the DAVIS regression run proved it: a second commit retrained all twelve). So:

1. **First commit:** leave `RESTORE_FROM = None` and commit.
2. When it finishes: open that version -> **Output** -> download `results`, and upload it
   as a Kaggle **Dataset** (e.g. `coldsite-grid36-results`).
3. **Every later commit:** attach that dataset (Add Input), set `RESTORE_FROM` in cell 5,
   and commit. Finished cells are skipped; interrupted ones are retrained.
4. After each commit, add the new output to the dataset (**New Version** of it), so the
   next restore has everything.

**The notebook stops itself at 11 hours.** Kaggle ends a commit at 12, and a commit cut off
mid-cell may not save its output. Stopping at 11 lets the commit finish cleanly and save
whatever completed. A cell stopped mid-training is retrained on the next commit.

## What to expect

| stage | per GPU | notes |
|---|---|---|
| 1. DeepDTA, 12 cells | ~0.5-1 h | fast; its AUROC is the sanity gate |
| 2. ColdSite-DTI, 6 cells per GPU | unknown until one finishes | DAVIS regression cells took 1-3.5 h each |
| 2. HyperAttentionDTI, 6 cells per GPU | ~3.5-9.5 h | STATUS.md estimate |

Realistically **2-3 commits**. The two GPUs split the work by split type -- GPU 0 takes
`random` and `cold_drug`, GPU 1 `cold_target` and `cold_pair` -- for all three models, so
both carry a similar load.


## 1. Check the GPUs


In [1]:
import time
START = time.time()          # the 11-hour self-stop is measured from here

import torch

assert torch.cuda.is_available(), 'No CUDA. Settings -> Accelerator -> GPU T4 x2.'
N_GPU = torch.cuda.device_count()
for i in range(N_GPU):
    p = torch.cuda.get_device_properties(i)
    print(f'GPU {i}: {p.name}, {p.total_memory/1e9:.1f} GB')
print('torch  :', torch.__version__)

# Measured peak memory on a 1000-residue protein (STATUS.md):
#   ColdSite-DTI  batch 64 -> 8.7 GB, 16 -> 2.3 GB
#   HyperAttentionDTI  batch 32 -> ~5 GB; 8 x accum 4 is the same effective batch
#   DeepDTA  batch 256 -> 0.7 GB
big = torch.cuda.get_device_properties(0).total_memory / 1e9 >= 14
COLDSITE_BATCH = 64 if big else 16
HAT_BATCH, HAT_ACCUM = (32, 1) if big else (8, 4)
DEEPDTA_BATCH = 256
print(f'batches: ColdSite {COLDSITE_BATCH}, HAT {HAT_BATCH}x{HAT_ACCUM}, DeepDTA {DEEPDTA_BATCH}')


GPU 0: Tesla T4, 15.6 GB
GPU 1: Tesla T4, 15.6 GB
torch  : 2.10.0+cu128
batches: ColdSite 64, HAT 32x1, DeepDTA 256


## 2. Clone the repo

Pinned to `main` on the fork. The cell refuses to continue on a checkout older than the
binary-label fix: before it, ColdSite-DTI's binary run crashed at its first AUROC because
the loader handed the model raw pKd values instead of classes.


In [2]:
import os

REPO = 'https://github.com/Mahim56207/ColdSite-DTI_New.git'
WORK = '/kaggle/working'
SRC  = f'{WORK}/ColdSite-DTI_New'

if not os.path.exists(SRC):
    !git clone --branch main {REPO} {SRC}
os.chdir(SRC)
!git pull origin main
!pip install -q tabulate subword-nmt

import importlib, src.model.dataset as _ds
importlib.reload(_ds)
assert hasattr(_ds, 'BINARY_THRESHOLD'), (
    'This checkout predates the binary-label fix -- ColdSite-DTI would crash. '
    'Re-run this cell so git pull fetches main.')

RESULTS = f'{WORK}/results'
os.makedirs(RESULTS, exist_ok=True)
print()
!git log --oneline -1
print('results ->', RESULTS)


Cloning into '/kaggle/working/ColdSite-DTI_New'...
remote: Enumerating objects: 963, done.
remote: Counting objects: 100% (217/217), done.
remote: Compressing objects: 100% (151/151), done.
remote: Total 963 (delta 120), reused 147 (delta 66), pack-reused 746 (from 1)
Receiving objects: 100% (963/963), 184.90 MiB | 25.96 MiB/s, done.
Resolving deltas: 100% (492/492), done.
From https://github.com/Mahim56207/ColdSite-DTI_New
 * branch            main       -> FETCH_HEAD
Already up to date.

c045e2c (HEAD -> main, origin/main, origin/HEAD) Replace the two-account KIBA plan with the six-account one
results -> /kaggle/working/results


## 3. Fetch the DeepDTA source files


In [3]:
BASE = 'https://raw.githubusercontent.com/hkmztrk/DeepDTA/master/data'
for ds in ('davis', 'kiba'):
    os.makedirs(f'src/data/baselines/deepdta/data/{ds}', exist_ok=True)
    for fname in ('ligands_can.txt', 'proteins.txt', 'Y'):
        target = f'src/data/baselines/deepdta/data/{ds}/{fname}'
        if not os.path.exists(target):
            !curl -sL {BASE}/{ds}/{fname} -o {target}
!python -m src.data.load_data


davis: 30056 measured pairs, 68 unique drugs, 442 unique targets, Y range [5.000, 10.796]
kiba: 118254 measured pairs, 2111 unique drugs, 229 unique targets, Y range [0.000, 17.200]


Expected, exactly: `davis: 30056 measured pairs, 68 unique drugs, 442 unique targets,
Y range [5.000, 10.796]`.


## 4. Build the splits — and verify they match

These 36 cells must come from the same splits as every other result in the project. The
counts below were verified on a MacBook, on Colab and on Kaggle; the cell asserts them.


In [4]:
!python -m src.data.build_splits 2>&1 | grep -E 'davis|leakage'

import pandas as pd

EXPECTED = {
    'random':      (21039, 3006, 6011),
    'cold_drug':   (21658, 2652, 5746),
    'cold_target': (21080, 2992, 5984),
    'cold_pair':   (15190,  264, 1144),
}
problems = []
for split, expected in EXPECTED.items():
    got = tuple(len(pd.read_csv(f'data/splits/davis/{split}/{part}.csv'))
                for part in ('train', 'valid', 'test'))
    if got != expected:
        problems.append(f'{split}: expected {expected}, got {got}')
    print(f"{split:12s} {str(got):26s} {'OK' if got == expected else 'MISMATCH'}")
assert not problems, 'Splits differ from the rest of the project:\n  ' + '\n  '.join(problems)
print('\nSplits match. Safe to train.')


=== Building splits for davis ===
Saved data/splits/davis/random/  train=21039  valid=3006  test=6011
Saved data/splits/davis/cold_drug/  train=21658  valid=2652  test=5746
Saved data/splits/davis/cold_target/  train=21080  valid=2992  test=5984
Saved data/splits/davis/cold_pair/  train=15190  valid=264  test=1144
cold_drug: no leakage across train/valid/test -- OK
cold_target: no leakage across train/valid/test -- OK
cold_pair: no leakage across train/valid/test -- OK
cold_drug: no leakage across train/valid/test -- OK
cold_target: no leakage across train/valid/test -- OK
cold_pair: no leakage across train/valid/test -- OK
random       (21039, 3006, 6011)        OK
cold_drug    (21658, 2652, 5746)        OK
cold_target  (21080, 2992, 5984)        OK
cold_pair    (15190, 264, 1144)         OK

Splits match. Safe to train.


## 5. Restore results from the previous commit

`None` on the first commit. After that, attach the dataset holding the previous output and
put its path here. Only model files are copied, and nothing already present is overwritten.


In [5]:
import shutil, glob

# e.g. '/kaggle/input/coldsite-grid36-results'
RESTORE_FROM = '/kaggle/input'
if RESTORE_FROM:
    assert os.path.isdir(RESTORE_FROM), f'not a directory: {RESTORE_FROM}'
    copied = skipped = 0
    for src in glob.glob(f'{RESTORE_FROM}/**/*', recursive=True):
        name = os.path.basename(src)
        if not name.endswith(('.pt', '_results.json', '_history.json')):
            continue
        dst = os.path.join(RESULTS, name)
        if os.path.exists(dst):
            skipped += 1
            continue
        shutil.copy2(src, dst)
        copied += 1
    print(f'restored {copied} file(s), left {skipped} already present')
else:
    print('RESTORE_FROM is None -- starting from an empty results folder.')


restored 65 file(s), left 0 already present


## 6. The runner

Each GPU works through its own queue of commands in order; the two GPUs run in parallel.
Output streams live, tagged `[GPU0]` / `[GPU1]`, and is teed to log files. At the 11-hour
mark it stops launching new work and ends anything still running, so the commit can save.


In [6]:
import subprocess, threading

DEADLINE = START + 11 * 3600
SPLITS = ['random', 'cold_drug', 'cold_target', 'cold_pair']
SEEDS = [1, 2, 3]
SHARE = ({'0': ['random', 'cold_drug'], '1': ['cold_target', 'cold_pair']}
         if N_GPU >= 2 else {'0': SPLITS})


def hours_left():
    return (DEADLINE - time.time()) / 3600


def run_parallel(queues, label):
    """queues: {gpu: [command, ...]}. Returns True if the deadline cut it short."""
    current, state = {}, {'deadline': False}

    def worker(gpu, commands):
        env = {**os.environ, 'CUDA_VISIBLE_DEVICES': gpu, 'PYTHONUNBUFFERED': '1'}
        with open(f'{WORK}/{label}_gpu{gpu}.log', 'a') as log:
            for cmd in commands:
                if time.time() > DEADLINE:
                    return
                proc = subprocess.Popen(cmd, env=env, stdout=subprocess.PIPE,
                                        stderr=subprocess.STDOUT, text=True, bufsize=1)
                current[gpu] = proc
                for line in proc.stdout:
                    print(f'[GPU{gpu}] {line}', end='', flush=True)
                    log.write(line)
                    log.flush()
                code = proc.wait()
                if code != 0 and not state['deadline']:
                    print(f'[GPU{gpu}] !! exited {code}: {" ".join(cmd[3:])}', flush=True)

    threads = [threading.Thread(target=worker, args=(g, q), daemon=True)
               for g, q in queues.items()]
    for t in threads:
        t.start()
    while any(t.is_alive() for t in threads):
        if time.time() > DEADLINE and not state['deadline']:
            state['deadline'] = True
            print('\n*** 11-hour mark: stopping so this commit can save its output. '
                  'Unfinished cells are retrained next commit. ***\n', flush=True)
            for proc in list(current.values()):
                if proc.poll() is None:
                    proc.terminate()
        time.sleep(15)
    return state['deadline']


def deepdta_cmd(split, seed):
    return ['python', '-u', '-m', 'src.model.train_deepdta',
            '--split-dir', f'data/splits/davis/{split}', '--dataset', 'davis',
            '--split', split, '--task', 'binary', '--seed', str(seed),
            '--batch-size', str(DEEPDTA_BATCH), '--min-epochs', '10', '--epochs', '100',
            '--checkpoint-dir', RESULTS, '--results-dir', RESULTS, '--skip-if-done']


def coldsite_cmd(splits):
    return ['python', '-u', '-m', 'src.model.run_grid', '--datasets', 'davis',
            '--splits', ','.join(splits), '--seeds', ','.join(map(str, SEEDS)),
            '--task', 'binary', '--epochs', '100', '--min-epochs', '10',
            '--batch-size', str(COLDSITE_BATCH), '--results-dir', RESULTS]


def hat_cmd(split, seed):
    return ['python', '-u', '-m', 'src.model.train_hyperattentiondti',
            '--split-dir', f'data/splits/davis/{split}', '--dataset', 'davis',
            '--split', split, '--seed', str(seed),
            '--batch-size', str(HAT_BATCH), '--accum-steps', str(HAT_ACCUM),
            '--min-epochs', '10', '--epochs', '100',
            '--checkpoint-dir', RESULTS, '--results-dir', RESULTS, '--skip-if-done']


for gpu, splits in SHARE.items():
    print(f'GPU {gpu} -> {", ".join(splits)}  (all three models)')
print(f'{hours_left():.1f} h left before the self-stop')


GPU 0 -> random, cold_drug  (all three models)
GPU 1 -> cold_target, cold_pair  (all three models)
11.0 h left before the self-stop


## 7. Stage 1 — DeepDTA

The cheapest model runs first, so a broken pipeline shows up in under an hour instead of
after twenty. Cells already finished (restored) are skipped in seconds.


In [7]:
stage1 = {gpu: [deepdta_cmd(split, seed) for split in splits for seed in SEEDS]
          for gpu, splits in SHARE.items()}
cut_short = run_parallel(stage1, 'deepdta')
print('\nDeepDTA stage', 'CUT SHORT by the deadline' if cut_short else 'done')


[GPU1] already done, skipping -> /kaggle/working/results/davis_cold_target_binary_seed1_deepdta_results.json
[GPU0] already done, skipping -> /kaggle/working/results/davis_random_binary_seed1_deepdta_results.json
[GPU1] already done, skipping -> /kaggle/working/results/davis_cold_target_binary_seed2_deepdta_results.json
[GPU0] already done, skipping -> /kaggle/working/results/davis_random_binary_seed2_deepdta_results.json
[GPU0] already done, skipping -> /kaggle/working/results/davis_random_binary_seed3_deepdta_results.json
[GPU1] already done, skipping -> /kaggle/working/results/davis_cold_target_binary_seed3_deepdta_results.json
[GPU1] already done, skipping -> /kaggle/working/results/davis_cold_pair_binary_seed1_deepdta_results.json
[GPU0] already done, skipping -> /kaggle/working/results/davis_cold_drug_binary_seed1_deepdta_results.json
[GPU0] already done, skipping -> /kaggle/working/results/davis_cold_drug_binary_seed2_deepdta_results.json
[GPU1] already done, skipping -> /kaggle

## 8. Sanity gate

DAVIS `random` AUROC should be well above chance and below perfect -- roughly 0.85-0.92.
**At 0.98 or above, something is leaking labels or the metric is wrong**, and spending
twenty more hours on ColdSite-DTI and HyperAttentionDTI would only produce more wrong
numbers. The gate does not raise (a failed commit may not save its output); it sets `GO`
and stage 2 refuses to start.


In [8]:
import json, statistics as st
from src.model.checkpoint_naming import checkpoint_path, results_path, run_tag


def auroc(model, split, seed):
    path = results_path(RESULTS, run_tag('davis', split, 'binary', seed), model=model)
    if not os.path.exists(path):
        return None
    return json.load(open(path))['test_metrics'].get('auroc')


GO, reasons = True, []
for split in SPLITS:
    values = [a for a in (auroc('deepdta', split, s) for s in SEEDS) if a is not None]
    shown = f'{st.mean(values):.4f} over {len(values)} seed(s)' if values else 'no results'
    print(f'DeepDTA {split:12s} AUROC {shown}')

random_values = [a for a in (auroc('deepdta', 'random', s) for s in SEEDS) if a is not None]
if not random_values:
    GO = False; reasons.append('no DeepDTA random-split result to check against')
elif st.mean(random_values) >= 0.98:
    GO = False; reasons.append(f'random AUROC {st.mean(random_values):.4f} is implausibly '
                               'high -- suspect label leakage or a metric bug')
elif st.mean(random_values) < 0.60:
    GO = False; reasons.append(f'random AUROC {st.mean(random_values):.4f} is near chance '
                               '-- the pipeline is not learning')

if GO:
    print('\nGate passed. Stage 2 will run.')
else:
    print('\n' + '!' * 70 + '\nGATE FAILED -- stage 2 will NOT run:\n  - '
          + '\n  - '.join(reasons) + '\n' + '!' * 70)


DeepDTA random       AUROC 0.9290 over 3 seed(s)
DeepDTA cold_drug    AUROC 0.6915 over 3 seed(s)
DeepDTA cold_target  AUROC 0.9075 over 3 seed(s)
DeepDTA cold_pair    AUROC 0.7277 over 3 seed(s)

Gate passed. Stage 2 will run.


## 9. Stage 2 — ColdSite-DTI, then HyperAttentionDTI

The long stage. Each GPU trains ColdSite-DTI on its two split types, then
HyperAttentionDTI on the same two. `run_grid` validates its first unfinished cell end to end
before launching the rest.


In [9]:
if not GO:
    print('Skipped: the sanity gate failed. See section 8.')
elif hours_left() < 0.5:
    print(f'Skipped: only {hours_left():.1f} h left. Commit again with the restore cell set.')
else:
    stage2 = {gpu: [coldsite_cmd(splits)] + [hat_cmd(split, seed)
                                             for split in splits for seed in SEEDS]
              for gpu, splits in SHARE.items()}
    cut_short = run_parallel(stage2, 'stage2')
    print('\nStage 2', 'CUT SHORT by the deadline -- commit again' if cut_short else 'done')


[GPU1] Grid: 6 cells, 6 unique run tags
[GPU1] warning: checkpoint already exists; the cell will be skipped if it is complete, retrained if it was interrupted: /kaggle/working/results/coldsite_dti_davis_cold_target_binary_seed1.pt
[GPU0] Grid: 6 cells, 6 unique run tags
[GPU1] warning: checkpoint already exists; the cell will be skipped if it is complete, retrained if it was interrupted: /kaggle/working/results/coldsite_dti_davis_cold_target_binary_seed2.pt
[GPU1] warning: checkpoint already exists; the cell will be skipped if it is complete, retrained if it was interrupted: /kaggle/working/results/coldsite_dti_davis_cold_target_binary_seed3.pt
[GPU0] warning: checkpoint already exists; the cell will be skipped if it is complete, retrained if it was interrupted: /kaggle/working/results/coldsite_dti_davis_random_binary_seed1.pt
[GPU1] warning: checkpoint already exists; the cell will be skipped if it is complete, retrained if it was interrupted: /kaggle/working/results/coldsite_dti_davi

## 10. What landed

A cell counts as complete only when both its checkpoint and its results file exist. A
checkpoint alone is an interrupted cell, and the next commit retrains it.


In [10]:
MODELS = ['deepdta', 'coldsite_dti', 'hyperattentiondti']
complete = interrupted = 0
print(f"{'model':18s} {'split':12s} {'cells':>5s}   AUROC mean +- sd")
for model in MODELS:
    for split in SPLITS:
        values = []
        for seed in SEEDS:
            ckpt = checkpoint_path(RESULTS, 'davis', split, 'binary', seed, model=model)
            res = results_path(RESULTS, run_tag('davis', split, 'binary', seed), model=model)
            if os.path.exists(ckpt) and os.path.exists(res):
                complete += 1
                values.append(json.load(open(res))['test_metrics']['auroc'])
            elif os.path.exists(ckpt):
                interrupted += 1
        spread = (f'{st.mean(values):.4f} +- {st.stdev(values):.4f}' if len(values) > 1
                  else f'{values[0]:.4f}' if values else '--')
        print(f'{model:18s} {split:12s} {len(values):>3d}/3   {spread}')

GRID_COMPLETE = complete == 36
print(f'\ncomplete    : {complete} / 36')
print(f'interrupted : {interrupted}  (retrained next commit)')
print(f'time left   : {hours_left():.1f} h')
print('\nALL 36 DONE -- analysis runs below.' if GRID_COMPLETE else
      '\nNot done yet: download the output, update the dataset, commit again with RESTORE_FROM set.')


model              split        cells   AUROC mean +- sd
deepdta            random         3/3   0.9290 +- 0.0023
deepdta            cold_drug      3/3   0.6915 +- 0.0437
deepdta            cold_target    3/3   0.9075 +- 0.0032
deepdta            cold_pair      3/3   0.7277 +- 0.0352
coldsite_dti       random         3/3   0.9240 +- 0.0012
coldsite_dti       cold_drug      3/3   0.7206 +- 0.0075
coldsite_dti       cold_target    3/3   0.8570 +- 0.0109
coldsite_dti       cold_pair      3/3   0.6236 +- 0.0991
hyperattentiondti  random         3/3   0.9370 +- 0.0049
hyperattentiondti  cold_drug      0/3   --
hyperattentiondti  cold_target    3/3   0.9148 +- 0.0015
hyperattentiondti  cold_pair      3/3   0.6939 +- 0.0375

complete    : 33 / 36
interrupted : 1  (retrained next commit)
time left   : -0.0 h

Not done yet: download the output, update the dataset, commit again with RESTORE_FROM set.


## 11. Analysis — only once all 36 cells exist

Faithfulness writes the accuracy file that the ladder refuses to draw without; then the
audit table (Holm-Bonferroni once across the whole family); then the non-kinase control,
reported with and without the cotransport-ion sites. DeepDTA is the accuracy anchor only --
it has no attention, so it is never audited.


In [11]:
if not GRID_COMPLETE:
    print('Skipped: the grid is incomplete.')
elif hours_left() < 1.5:
    print(f'Skipped: {hours_left():.1f} h left is not enough. The grid is complete, so the '
          'next commit will go straight to this cell.')
else:
    GT = 'data/davis_ground_truth_sites.json'
    for seed in SEEDS:
        print(f'\n===== faithfulness + ladder, seed {seed} =====')
        !python -m src.evaluation.run_faithfulness --dataset davis --seed {seed} --task binary \
            --checkpoint-dir {RESULTS} --results-dir {RESULTS} --out-dir {RESULTS}
        !python -m src.evaluation.run_ladder --dataset davis --seed {seed} --task binary \
            --ground-truth {GT} --checkpoint-dir {RESULTS} --out-dir {RESULTS} \
            --accuracy-json {RESULTS}/accuracy_davis_seed{seed}.json

    print('\n===== audit grid =====')
    !python -m src.evaluation.run_audit --models coldsite_dti,hyperattentiondti,uniform_control \
        --datasets davis --seeds 1,2,3 --task binary --ground-truth {GT} \
        --checkpoint-dir {RESULTS} --out-dir {RESULTS}

    for model in ('coldsite_dti', 'hyperattentiondti'):
        for extra in ('', '--exclude-cotransport-ions'):
            print(f'\n===== control: {model} {extra or "(all ligands)"} =====')
            !python -m src.evaluation.run_control --model {model} --dataset davis --seed 1 \
                --task binary --checkpoint-dir {RESULTS} --out-dir {RESULTS} {extra}


Skipped: the grid is incomplete.


## 12. Take the results with you

Everything in `/kaggle/working` is saved as this version's output. The zip is the easy
download. Add its contents to your restore dataset (as a new version) before the next
commit, and keep a copy in Google Drive -- the checkpoints are what the analysis reads, and
retraining them costs the whole grid again.


In [12]:
!cd {WORK} && rm -f grid36_results.zip && zip -qr grid36_results.zip results
print(f'{WORK}/grid36_results.zip  --  download from the Output panel')


/kaggle/working/grid36_results.zip  --  download from the Output panel
